# Lab 4 - Urban and Forest Classification

The task is to classify a `200 x 200` Sentinel-2 image patch as forest or urban.

## 1. Task as a function

The classifier is a function:

`f(image patch) -> class`

The input is a `200 x 200 x 3` RGB patch. The output is `0` for forest or `1` for urban.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

data_folder = Path('satellite_data')
if not data_folder.exists():
    data_folder = Path('lab4/satellite_data')

forest = np.array(Image.open(data_folder / 'forest_amazon.png').convert('RGB'))
urban = np.array(Image.open(data_folder / 'urban_riyadh.png').convert('RGB'))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(forest)
axes[0].set_title('Forest')
axes[1].imshow(urban)
axes[1].set_title('Urban')
for ax in axes:
    ax.axis('off')
plt.show()

## 2. Training data

Non-overlapping `200 x 200` patches are taken from both images.

In [ ]:
PATCH_SIZE = 200
rng = np.random.default_rng(42)

def make_patches(image):
    patches = []
    for row in range(5):
        for column in range(5):
            y = row * PATCH_SIZE
            x = column * PATCH_SIZE
            patches.append(image[y:y + PATCH_SIZE, x:x + PATCH_SIZE])
    return np.array(patches).astype('float32') / 255.0

forest_patches = make_patches(forest)
urban_patches = make_patches(urban)

X_train = np.concatenate([forest_patches[:15], urban_patches[:15]])
y_train = np.concatenate([np.zeros(15), np.ones(15)])
X_test = np.concatenate([forest_patches[15:], urban_patches[15:]])
y_test = np.concatenate([np.zeros(10), np.ones(10)])

train_order = rng.permutation(len(X_train))
test_order = rng.permutation(len(X_test))
X_train, y_train = X_train[train_order], y_train[train_order]
X_test, y_test = X_test[test_order], y_test[test_order]

print('Training patches:', len(X_train))
print('Test patches:', len(X_test))
print('Patch shape:', X_train[0].shape)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, patch, label in zip(axes.ravel(), X_train[:8], y_train[:8]):
    ax.imshow(patch)
    ax.set_title('Urban' if label == 1 else 'Forest')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. ANN architecture

A small convolutional neural network is used. Convolutional layers find image patterns, and the final sigmoid neuron returns the probability of the urban class.

In [ ]:
def build_model(filters=(8, 16), dense_nodes=8, optimizer='adam'):
    model = tf.keras.Sequential([tf.keras.layers.Input((200, 200, 3))])
    for number in filters:
        model.add(tf.keras.layers.Conv2D(number, 3, activation='relu'))
        model.add(tf.keras.layers.MaxPooling2D())
    model.add(tf.keras.layers.GlobalAveragePooling2D())
    model.add(tf.keras.layers.Dense(dense_nodes, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

## 4. Five experiments

In [ ]:
experiments = [
    ('1: small CNN', (8,), 8, 'adam'),
    ('2: two layers', (8, 16), 8, 'adam'),
    ('3: more filters', (16, 32), 16, 'adam'),
    ('4: SGD', (8, 16), 8, 'sgd'),
    ('5: three layers', (8, 16, 32), 16, 'adam')
]

results = []
models = []

for name, filters, nodes, optimizer in experiments:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    model = build_model(filters, nodes, optimizer)
    model.fit(X_train, y_train, epochs=5, batch_size=8, verbose=0)
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    results.append((name, accuracy))
    models.append(model)
    print(f'{name}: {accuracy:.2%}')

## 5. Results

In [ ]:
names = [result[0] for result in results]
accuracies = [result[1] for result in results]

plt.figure(figsize=(9, 4))
plt.bar(names, accuracies)
plt.ylim(0, 1)
plt.ylabel('Test accuracy')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

best = int(np.argmax(accuracies))
print('Best model:', names[best])
print('Best accuracy:', f'{accuracies[best]:.2%}')

In [ ]:
best_model = models[best]
predictions = (best_model.predict(X_test, verbose=0).ravel() >= 0.5).astype(int)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, patch, true, predicted in zip(axes.ravel(), X_test[:8], y_test[:8], predictions[:8]):
    ax.imshow(patch)
    ax.set_title(f'True: {int(true)}, Pred: {predicted}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Conclusion

The experiments compare five simple CNN configurations. The printed results show how filters, layers and the optimizer affect accuracy. The dataset is still small, so more independent satellite images would be needed for a reliable real-world classifier.